<a href="https://colab.research.google.com/github/ganja025/ganja025/blob/main/qwythos_ULTIMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Llama-Server with Ngrok and Drive Integration

In [8]:
!pip install -qqq pyngrok

In [16]:
import os
import subprocess
import time
import shutil
from google.colab import drive
from pyngrok import ngrok, conf
from google.colab import userdata # Import userdata to access secrets

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Install pyngrok if not already installed
!pip install -qqq pyngrok

# 3. Define Constants
MODEL_URL = 'https://huggingface.co/empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF/resolve/main/Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.gguf?download=true'
SERVER_PORT = 8889  # You can change this port if needed

# Path to the llama-server executable within your Drive folder
# Adjust this path if your llama-server executable is in a different subdirectory
llama_cpp_binary_dir_drive = '/content/drive/MyDrive/Colab Notebooks/cuda'

# 4. Configure ngrok authentication token
# NGROK_TOKEN is assumed to be available from kernel state
NGROK_TOKEN = userdata.get('NGROK_TOKEN') # Fetch NGROK_TOKEN from Colab secrets
if not NGROK_TOKEN:
    raise ValueError("NGROK_TOKEN not found in Colab secrets. Please add it to your Colab secrets.")
conf.get_default().auth_token = NGROK_TOKEN

# Define paths for the model and temporary local directory
drive_path = '/content/drive/MyDrive/'
model_filename = os.path.basename(MODEL_URL.split('?')[0])
model_save_path = os.path.join(drive_path, model_filename)

local_tmp_dir = '/tmp'
local_llama_server_path = os.path.join(local_tmp_dir, 'llama-server')

# Ensure the model directory exists
os.makedirs(drive_path, exist_ok=True)

# 5. Download the model (if not already downloaded)
if not os.path.exists(model_save_path):
    print(f"Downloading model from {MODEL_URL} to {model_save_path}")
    !wget -O "{model_save_path}" "{MODEL_URL}"
    print("Download complete!")
else:
    print(f"Model already exists at {model_save_path}, skipping download.")

# 6. Copy the executable and its dependencies to a temporary local directory
if not os.path.exists(llama_cpp_binary_dir_drive):
    raise FileNotFoundError(f"The directory '{llama_cpp_binary_dir_drive}' was not found in Google Drive. Please verify the path.")

print(f"Copying content of '{llama_cpp_binary_dir_drive}' to '{local_tmp_dir}'...")

for item in os.listdir(llama_cpp_binary_dir_drive):
    s = os.path.join(llama_cpp_binary_dir_drive, item)
    d = os.path.join(local_tmp_dir, item)
    try:
        if os.path.isdir(s):
            shutil.copytree(s, d, symlinks=True, ignore_dangling_symlinks=True)
        else:
            shutil.copy2(s, d)
        print(f"Copied: {item}")
    except shutil.Error as e:
        print(f"Warning: Could not copy {s} to {d}: {e}")

# 7. Function to create generic symlinks for libraries
def create_symlink_if_needed(specific_lib_name, generic_lib_name):
    specific_path = os.path.join(local_tmp_dir, specific_lib_name)
    generic_path = os.path.join(local_tmp_dir, generic_lib_name)

    if os.path.exists(specific_path):
        if not os.path.exists(generic_path):
            try:
                os.symlink(specific_path, generic_path)
                print(f"Created symlink: {generic_path} -> {specific_path}")
            except OSError as e:
                print(f"Error creating symlink {generic_path} -> {specific_path}: {e}")
        else:
            print(f"Symlink {generic_path} already exists.")
    else:
        print(f"Warning: {specific_path} does not exist, cannot create symlink for {generic_lib_name}")

# Create symlinks for common libraries
create_symlink_if_needed('libllama-common.so.0.0.1', 'libllama-common.so.0')
create_symlink_if_needed('libmtmd.so.0.0.1', 'libmtmd.so.0')
create_symlink_if_needed('libllama.so.0.0.1', 'libllama.so.0')
create_symlink_if_needed('libggml.so.0.15.3', 'libggml.so.0')
create_symlink_if_needed('libggml-base.so.0.15.3', 'libggml-base.so.0')
create_symlink_if_needed('libggml-cpu.so.0.15.3', 'libggml-cpu.so.0')
create_symlink_if_needed('libggml-cuda.so.0.15.3', 'libggml-cuda.so.0')

# Ensure the local executable has correct permissions
if os.path.exists(local_llama_server_path):
    os.chmod(local_llama_server_path, 0o755)
    print(f"The local executable '{local_llama_server_path}' now has execution permissions.")
else:
    raise FileNotFoundError(f"The llama-server executable was not found in '{local_tmp_dir}' after copying. Ensure it exists in '{llama_cpp_binary_dir_drive}'.")

# 8. Determine CUDA Library Path Dynamically
cuda_lib_path = ""
for version in ['13.0', '12', '11']:
    potential_path = f"/usr/local/cuda-{version}/lib64"
    if os.path.exists(potential_path):
        cuda_lib_path = potential_path
        print(f"Found CUDA library path: {cuda_lib_path}")
        break
if not cuda_lib_path and os.path.exists("/usr/local/cuda/lib64"):
    cuda_lib_path = "/usr/local/cuda/lib64"
    print(f"Found generic CUDA library path: {cuda_lib_path}")
elif not cuda_lib_path:
    print("Warning: Could not find a specific CUDA library path. Proceeding without it, which might cause issues.")

# 9. Configure LD_LIBRARY_PATH for the subprocess
my_env = os.environ.copy()
ld_library_paths = [local_tmp_dir]
if cuda_lib_path:
    ld_library_paths.append(cuda_lib_path)

existing_ld_path = my_env.get("LD_LIBRARY_PATH", "")
if existing_ld_path:
    ld_library_paths.append(existing_ld_path)

my_env["LD_LIBRARY_PATH"] = os.pathsep.join(ld_library_paths)

print(f"LD_LIBRARY_PATH configured for the subprocess: {my_env.get('LD_LIBRARY_PATH')}")

# 10. Clear old logs before starting
log_stderr_path = '/content/llama_server_stderr.log'
log_stdout_path = '/content/llama_server_stdout.log'

for log_path in [log_stderr_path, log_stdout_path]:
    if os.path.exists(log_path):
        os.remove(log_path)
        print(f"Removed old log: {log_path}")

# 11. Start llama-server process
print(f"Starting llama-server on port {SERVER_PORT} with model {model_save_path}")

llama_server_command = [
    local_llama_server_path,
    '--model', model_save_path,
    '--host', '0.0.0.0',
    '--port', str(SERVER_PORT),
    '-c', '16384'  # Context size, adjust as needed
]

# Open log files in write mode
llama_server_stdout_file = open(log_stdout_path, 'w')
llama_server_stderr_file = open(log_stderr_path, 'w')

llama_process = subprocess.Popen(
    llama_server_command,
    stdout=llama_server_stdout_file,
    stderr=llama_server_stderr_file,
    env=my_env
)

print(f"Llama-server process started with PID: {llama_process.pid}")
print(f"Logs are being written to {log_stdout_path} and {log_stderr_path}")

# Give the server some time to start up
print("Waiting 120 seconds for llama-server to initialize...")
time.sleep(120)  # Adjust based on model loading time

# Check if server is running
if llama_process.poll() is not None:
    print("Llama-server process terminated prematurely. Checking logs...")
    with open(log_stderr_path, 'r') as f:
        print("\n--- llama_server_stderr.log content ---")
        print(f.read())
    with open(log_stdout_path, 'r') as f:
        print("\n--- llama_server_stdout.log content ---")
        print(f.read())
    raise RuntimeError("Llama-server failed to start.")

# 12. Clear any existing ngrok tunnels to avoid errors
def clear_ngrok_tunnels():
    try:
        tunnels = ngrok.get_tunnels()
        for tunnel in tunnels:
            print(f"Closing existing ngrok tunnel: {tunnel.public_url}")
            ngrok.disconnect(tunnel.public_url)
    except Exception as e:
        print(f"Could not close existing ngrok tunnels: {e}")

clear_ngrok_tunnels()

# 13. Open ngrok tunnel
print("Attempting to establish ngrok tunnel...")
try:
    public_url = ngrok.connect(SERVER_PORT)
    print(f"¡Ngrok tunnel established! Access your llama-server at: {public_url}")
except Exception as e:
    print(f"Error establishing ngrok tunnel: {e}")
    print("Please check your ngrok token and ensure the llama-server is running correctly.")
    # Attempt to read logs if ngrok failed
    with open(log_stderr_path, 'r') as f:
        print("\n--- llama_server_stderr.log content after ngrok attempt ---")
        print(f.read())
    with open(log_stdout_path, 'r') as f:
        print("\n--- llama_server_stdout.log content after ngrok attempt ---")
        print(f.read())

print("Remember to close the ngrok tunnel and the llama-server process when you are finished.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model already exists at /content/drive/MyDrive/Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.gguf, skipping download.
Copying content of '/content/drive/MyDrive/Colab Notebooks/cuda' to '/tmp'...
Copied: llama-cli
Copied: llama-completion
Copied: llama-gemma3-cli
Copied: libllama-completion-impl.so
Copied: libllama-batched-bench-impl.so
Copied: libllama-common.so.0.0.1
Copied: llama-bench
Copied: llama-imatrix
Copied: llama-mtmd-cli
Copied: libllama-fit-params-impl.so
Copied: libggml-cuda.so.0.15.3
Copied: llama-results
Copied: libmtmd.so.0.0.1
Copied: llama-tts
Copied: llama-export-lora
Copied: libllama-server-impl.so
Copied: llama-tokenize
Copied: llama-server
Copied: libggml.so.0.15.3
Copied: llama-gguf-split
Copied: libggml-cpu.so.0.15.3
Copied: libllama-bench-impl.so
Copied: libllama.so.0.0.1
Copied: libllama-perplexity-impl.so
Copied: llama-template-analysis
Cop

## Cleanup (Optional)

Run the following cell to properly shut down the `llama-server` process and close the ngrok tunnel when you are done. This is important to free up resources and avoid issues with ngrok tunnel limits.

In [15]:
# Cell to clean up resources

print("Cleaning up resources...")

# Terminate llama-server process if it's still running
if 'llama_process' in locals() and llama_process.poll() is None:
    print(f"Terminating llama-server process with PID: {llama_process.pid}")
    llama_process.terminate()
    llama_process.wait(timeout=10) # Give it some time to terminate gracefully
    if llama_process.poll() is None:
        print("Llama-server did not terminate gracefully, killing process.")
        llama_process.kill()

# Close ngrok tunnels
try:
    ngrok.kill()
    print("Ngrok tunnels closed.")
except Exception as e:
    print(f"Error closing ngrok tunnels: {e}")

print("Cleanup complete.")

# Close log files if they were opened
if 'llama_server_stdout_file' in locals() and not llama_server_stdout_file.closed:
    llama_server_stdout_file.close()
if 'llama_server_stderr_file' in locals() and not llama_server_stderr_file.closed:
    llama_server_stderr_file.close()


Cleaning up resources...
Terminating llama-server process with PID: 6164
Ngrok tunnels closed.
Cleanup complete.


DE AKI PABAJO MIERDA TODO


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
MODEL_URL = 'https://huggingface.co/empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF/resolve/main/Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.gguf?download=true'

In [ ]:
import os

# Define the target path in Google Drive
drive_path = '/content/drive/MyDrive/'
model_filename = os.path.basename(MODEL_URL.split('?')[0])
model_save_path = os.path.join(drive_path, model_filename)

# Ensure the directory exists
os.makedirs(drive_path, exist_ok=True)

print(f"Downloading model from {MODEL_URL} to {model_save_path}")
!wget -O "{model_save_path}" "{MODEL_URL}"
print("Download complete!")

--2026-07-09 22:13:35--  https://huggingface.co/empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF/resolve/main/Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.gguf?download=true
Resolving huggingface.co (huggingface.co)... 18.239.50.16, 18.239.50.103, 18.239.50.80, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.16|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6a3552eea519cd3014351343/b9bc7107662e61fa1732742dc3d27151ea34f97de6ac01ba001944ff9f3e283a?user_id=public&response-content-disposition=attachment%3B+filename*%3DUTF-8%27%27Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.gguf%3B+filename%3D%22Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.gguf%22%3B&X-Xet-Cas-Uid=public&Expires=1783638815&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmEzNTUyZWVhNTE5Y2QzMDE0MzUxMzQzL2I5YmM3MTA3NjYyZTYxZmExNzMyNzQyZGMzZDI3MTUxZWEzNGY5N2RlNmFjMDFiYTAwMTk0NGZmOWYzZTI4M2FcXD91c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1j

In [1]:
SERVER_PORT = 8889 # Puedes cambiar este puerto si es necesario

In [17]:
with open('llama_server_stderr.log', 'r') as f:
    stderr_content = f.read()
print(stderr_content)

0.00.221.023 I cmn  common_param: common_params_print_info: verbosity = 3 (adjust with the `-lv N` CLI arg)
0.00.429.404 I srv    load_model: loading model '/content/drive/MyDrive/Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.gguf'
0.26.385.685 I srv    load_model: initializing, n_slots = 4, n_ctx_slot = 16384, kv_unified = 'true'
0.26.447.746 I srv          init: chat template supports preserving reasoning, consider enabling it via --reasoning-preserve
0.26.447.821 I srv  llama_server: model loaded
0.26.447.835 I srv  llama_server: listening on http://0.0.0.0:8889
3.59.673.085 I slot get_availabl: id  3 | task -1 | selected slot by LRU, t_last = -1
3.59.673.220 I slot launch_slot_: id  3 | task 0 | processing task, is_child = 0
4.00.122.981 I slot get_availabl: id  2 | task -1 | selected slot by LRU, t_last = -1
4.00.127.104 I slot launch_slot_: id  2 | task 2 | processing task, is_child = 0
4.04.643.989 I slot print_timing: id  2 | task 2 | prompt processing, n_tokens =   4096, progress = 0.4